# Notebook 01 — Collect and Prepare the Six Botschaften

## Purpose

This notebook is the first stage of the thesis data pipeline. It creates a reproducible local corpus of the six selected **German-language Swiss Federal Council memorandums (Botschaften)** that will later be used for question-set construction, answer generation, and source-grounded evaluation.

It includes:

1. executing the Fedlex SPARQL discovery command to create the query-results CSV;
2. loading and checking the downloaded `queryResults.csv` file;
3. selecting **exactly the six documents chosen for the thesis**;
4. extracting their PDF URLs from the CSV rather than hard-coding download URLs;
5. generating and executing `curl` commands to download the six PDFs;
6. verifying the downloaded files;
7. extracting text from each PDF page-by-page;
8. saving page-level and document-level text/JSON representations; and
9. producing a manifest for later notebooks.

### Selected documents

- **Keine 10-Millionen-Schweiz! (Nachhaltigkeitsinitiative)**
- **Neutralitätsinitiative**
- **Stopfleber-Initiative**
- **Umweltverantwortungsinitiative**
- **Steuergerechtigkeits-Initiative**
- **13. AHV-Rente**

The notebook downloads `queryResults.csv` directly from the Fedlex SPARQL endpoint before using it as the source of truth for the PDF URLs.

## 0. Reproduce the Fedlex SPARQL query

This cell executes the Fedlex discovery query and saves the result as `queryResults.csv` in the
current project directory. The query searches Fedlex for German-language documents of resource
type `23`, restricted to 2022–2025 and titles containing `Volksinitiative`.

The notebook is therefore self-contained: no manual download or copying of the query-results CSV
is required before running the later cells.


In [21]:
# Download the Fedlex SPARQL query results.
# The result is saved as queryResults.csv and is used by the subsequent cells.

from pathlib import Path
import subprocess

PROJECT_ROOT = Path.cwd()
CSV_PATH = (PROJECT_ROOT / "../data/manifests/queryResults.csv").resolve()
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

curl_command = r"""curl 'https://fedlex.data.admin.ch/sparqlendpoint' \
  -X POST \
  --data-urlencode 'query=PREFIX jolux: <http://data.legilux.public.lu/resource/ontology/jolux#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT DISTINCT ?date ?title ?doc ?responsibleBody ?responsibleBodyLabel ?pdfURL
WHERE {
  ?doc jolux:typeDocument <https://fedlex.data.admin.ch/vocabulary/resource-type/23> ;
       jolux:publicationDate ?date ;
       jolux:isRealizedBy ?germanVersion ;
       jolux:responsibilityOf ?responsibleBody .

  ?germanVersion jolux:language <http://publications.europa.eu/resource/authority/language/DEU> ;
                 jolux:title ?title ;
                 jolux:isEmbodiedBy ?manifestation .

  ?manifestation jolux:format <http://publications.europa.eu/resource/authority/file-type/PDF> ;
                 jolux:isExemplifiedBy ?pdfURL .

  ?responsibleBody skos:prefLabel ?responsibleBodyLabel .

  FILTER(?date >= "2022-01-01"^^xsd:date && ?date < "2026-01-01"^^xsd:date)
  FILTER(CONTAINS(LCASE(STR(?title)), "volksinitiative"))
  FILTER(LANG(?responsibleBodyLabel) = "de")
}
ORDER BY ?date' \
  -H 'Accept: text/csv' \
  -o '""" + str(CSV_PATH) + """'"""

print("Downloading Fedlex SPARQL results to:", CSV_PATH)
subprocess.run(
    curl_command,
    shell=True,
    cwd=PROJECT_ROOT,
    check=True
)

if not CSV_PATH.exists() or CSV_PATH.stat().st_size == 0:
    raise RuntimeError(f"Fedlex query did not produce a valid CSV: {CSV_PATH}")

print(f"Downloaded query-results CSV: {CSV_PATH}")
print(f"File size: {CSV_PATH.stat().st_size:,} bytes")

Downloaded query-results CSV: /home/bern/Documents/llm_trustworthiness/data/manifests/queryResults.csv
File size: 9,961 bytes


  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100  11346 100   9961 100   1385 107.8k  15357                              0


## 1. Imports and project directories

The notebook creates a predictable directory structure. The Fedlex query-results CSV is downloaded
automatically in Section 0 as `queryResults.csv`.

```text
project/
├── queryResults.csv
├── data/
│   ├── pdfs/
│   ├── extracted_text/
│   └── manifests/
└── scripts/
    └── download_selected_botschaften.sh
```

The downloaded CSV is not modified after retrieval.


In [23]:
from pathlib import Path
import json
import re
import subprocess
import shlex
from datetime import datetime, timezone

import pandas as pd

PROJECT_ROOT = Path.cwd()

CSV_PATH = (PROJECT_ROOT / "../data/manifests/queryResults.csv").resolve()

DATA_DIR = (PROJECT_ROOT / "../data").resolve()
PDF_DIR = DATA_DIR / "pdfs"
TEXT_DIR = DATA_DIR / "extracted_text"
MANIFEST_DIR = DATA_DIR / "manifests"
SCRIPTS_DIR = (PROJECT_ROOT / "../scripts").resolve()

for directory in [PDF_DIR, TEXT_DIR, MANIFEST_DIR, SCRIPTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("CSV:", CSV_PATH)
print("PDF directory:", PDF_DIR)
print("Text directory:", TEXT_DIR)
print("Manifest directory:", MANIFEST_DIR)
print("Scripts directory:", SCRIPTS_DIR)

CSV: /home/bern/Documents/llm_trustworthiness/data/manifests/queryResults.csv
PDF directory: /home/bern/Documents/llm_trustworthiness/data/pdfs
Text directory: /home/bern/Documents/llm_trustworthiness/data/extracted_text
Manifest directory: /home/bern/Documents/llm_trustworthiness/data/manifests
Scripts directory: /home/bern/Documents/llm_trustworthiness/scripts


## 2. Define the six selected documents

The code uses explicit title fragments to identify the six intended records in the existing SPARQL results. It then checks that exactly one row matches each fragment.

In [24]:
SELECTED_TITLE_FRAGMENTS = {
    "nachhaltigkeitsinitiative": "Keine 10-Millionen-Schweiz!",
    "neutralitaetsinitiative": "Neutralitätsinitiative",
    "stopfleber_initiative": "Stopfleber-Initiative",
    "umweltverantwortungsinitiative": "Umweltverantwortungsinitiative",
    "steuergerechtigkeits_initiative": "Steuergerechtigkeits-Initiative",
    "13_ahv_rente": "13. AHV-Rente",
}

EXPECTED_COUNT = len(SELECTED_TITLE_FRAGMENTS)
print(f"Expected selected documents: {EXPECTED_COUNT}")

Expected selected documents: 6


## 3. Load the existing query-results CSV

The exported CSV contains spaces around fields, so `skipinitialspace=True` is used. The trailing unnamed column is removed if present.

In [25]:
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {CSV_PATH}. Check the Fedlex download cell and CSV_PATH."
    )

query_results = pd.read_csv(CSV_PATH, skipinitialspace=True)
query_results.columns = [c.strip() for c in query_results.columns]
query_results = query_results.loc[:, ~query_results.columns.str.startswith("Unnamed")]

required_columns = {"date", "title", "doc", "responsibleBody", "responsibleBodyLabel", "pdfURL"}
missing = required_columns - set(query_results.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {sorted(missing)}")

for column in required_columns:
    query_results[column] = query_results[column].astype(str).str.strip()

print(f"Rows in downloaded CSV: {len(query_results)}")
query_results[["date", "title", "doc", "pdfURL"]].head()

Rows in downloaded CSV: 23


,date,title,doc,pdfURL
0,2022-03-30,Botschaft zur Volksinitiative «Für die Zukunft...,https://fedlex.data.admin.ch/eli/fga/2022/737,https://fedlex.data.admin.ch/filestore/fedlex....
1,2022-06-21,Botschaft zur Volksinitiative «Für ein bessere...,https://fedlex.data.admin.ch/eli/fga/2022/1485,https://fedlex.data.admin.ch/filestore/fedlex....
2,2022-07-13,Botschaft zur Volksinitiative «Für eine sicher...,https://fedlex.data.admin.ch/eli/fga/2022/1711,https://fedlex.data.admin.ch/filestore/fedlex....
3,2023-01-12,Botschaft zur Volksinitiative «Für Freiheit un...,https://fedlex.data.admin.ch/eli/fga/2023/59,https://fedlex.data.admin.ch/filestore/fedlex....
4,2024-01-19,Botschaft zur Volksinitiative «Für eine verant...,https://fedlex.data.admin.ch/eli/fga/2024/109,https://fedlex.data.admin.ch/filestore/fedlex....


## 4. Select exactly the six documents from the CSV

The PDF URLs are **not hard-coded**. They are taken directly from the matching rows in `queryResults.csv`.

The validation below stops the notebook if a title cannot be found, if multiple rows match a title, if a PDF URL is missing, or if duplicate document URIs are selected.

In [26]:
selected_rows = []

for key, fragment in SELECTED_TITLE_FRAGMENTS.items():
    matches = query_results[
        query_results["title"].str.contains(fragment, case=False, regex=False, na=False)
    ]

    if len(matches) == 0:
        raise ValueError(f"No CSV row found for: {fragment}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple CSV rows found for '{fragment}'.\n" +
            matches[["date", "title", "doc", "pdfURL"]].to_string(index=False)
        )

    row = matches.iloc[0].copy()
    row["selection_key"] = key
    row["selection_fragment"] = fragment
    selected_rows.append(row)

selected = pd.DataFrame(selected_rows).reset_index(drop=True)

assert len(selected) == EXPECTED_COUNT
assert selected["doc"].nunique() == EXPECTED_COUNT, "Duplicate document URI detected."
assert selected["pdfURL"].notna().all(), "A selected document has no PDF URL."
assert selected["pdfURL"].str.startswith("http").all(), "A selected PDF URL is invalid."

selected = selected[
    ["selection_key", "selection_fragment", "date", "title", "doc",
     "responsibleBody", "responsibleBodyLabel", "pdfURL"]
]

selected

,selection_key,selection_fragment,date,title,doc,responsibleBody,responsibleBodyLabel,pdfURL
0,nachhaltigkeitsinitiative,Keine 10-Millionen-Schweiz!,2025-04-11,Botschaft zur Volksinitiative «Keine 10-Millio...,https://fedlex.data.admin.ch/eli/fga/2025/1262,https://fedlex.data.admin.ch/vocabulary/legal-...,Staatssekretariat für Migration,https://fedlex.data.admin.ch/filestore/fedlex....
1,neutralitaetsinitiative,Neutralitätsinitiative,2024-12-18,Botschaft zur Volksinitiative «Wahrung der sch...,https://fedlex.data.admin.ch/eli/fga/2024/3136,https://fedlex.data.admin.ch/vocabulary/legal-...,Direktion für Völkerrecht,https://fedlex.data.admin.ch/filestore/fedlex....
2,stopfleber_initiative,Stopfleber-Initiative,2024-12-12,Botschaft zur Volksinitiative «Ja zum Importve...,https://fedlex.data.admin.ch/eli/fga/2024/3077,https://fedlex.data.admin.ch/vocabulary/legal-...,Bundesamt für Lebensmittelsicherheit und Veter...,https://fedlex.data.admin.ch/filestore/fedlex....
3,umweltverantwortungsinitiative,Umweltverantwortungsinitiative,2024-01-19,Botschaft zur Volksinitiative «Für eine verant...,https://fedlex.data.admin.ch/eli/fga/2024/109,https://fedlex.data.admin.ch/vocabulary/legal-...,Bundesamt für Umwelt,https://fedlex.data.admin.ch/filestore/fedlex....
4,steuergerechtigkeits_initiative,Steuergerechtigkeits-Initiative,2024-03-15,Botschaft zur Volksinitiative «Für eine zivils...,https://fedlex.data.admin.ch/eli/fga/2024/589,https://fedlex.data.admin.ch/vocabulary/legal-...,Eidgenössische Steuerverwaltung,https://fedlex.data.admin.ch/filestore/fedlex....
5,13_ahv_rente,13. AHV-Rente,2022-06-21,Botschaft zur Volksinitiative «Für ein bessere...,https://fedlex.data.admin.ch/eli/fga/2022/1485,https://fedlex.data.admin.ch/vocabulary/legal-...,Bundesamt für Sozialversicherungen,https://fedlex.data.admin.ch/filestore/fedlex....


## 5. Save the selected-document manifest

This manifest records the exact six records selected from the original SPARQL output and becomes the input to the download/extraction stages.

In [27]:
selection_manifest_path = MANIFEST_DIR / "selected_botschaften.csv"
selected.to_csv(selection_manifest_path, index=False, encoding="utf-8")
print(f"Saved: {selection_manifest_path}")

Saved: /home/bern/Documents/llm_trustworthiness/data/manifests/selected_botschaften.csv


## 6. Generate the `curl` download commands

For every selected record, the notebook generates a real command such as:

```bash
curl -L --fail --retry 3 --retry-delay 2 -o data/pdfs/<filename>.pdf "<PDF URL>"
```

The URL always comes from the CSV. The output filename is generated locally from the selection key and Fedlex document URI.

In [28]:
def safe_filename(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._")


def document_filename(row) -> str:
    # Example: 13_ahv_rente_fga_2022_1485.pdf
    parts = row["doc"].rstrip("/").split("/")
    suffix = "_".join(safe_filename(x) for x in parts[-2:])
    return f"{safe_filename(row['selection_key'])}_{suffix}.pdf"


download_records = []
for _, row in selected.iterrows():
    output_path = PDF_DIR / document_filename(row)
    cmd = [
        "curl", "-L", "--fail", "--retry", "3", "--retry-delay", "2",
        "-o", str(output_path), row["pdfURL"]
    ]

    download_records.append({
        "selection_key": row["selection_key"],
        "title": row["title"],
        "doc": row["doc"],
        "pdfURL": row["pdfURL"],
        "output_path": str(output_path),
        "command": shlex.join(cmd),
        "cmd": cmd,
    })

for record in download_records:
    print(record["command"])

curl -L --fail --retry 3 --retry-delay 2 -o /home/bern/Documents/llm_trustworthiness/data/pdfs/nachhaltigkeitsinitiative_2025_1262.pdf https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/fga/2025/1262/de/pdf-a/fedlex-data-admin-ch-eli-fga-2025-1262-de-pdf-a.pdf
curl -L --fail --retry 3 --retry-delay 2 -o /home/bern/Documents/llm_trustworthiness/data/pdfs/neutralitaetsinitiative_2024_3136.pdf https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/fga/2024/3136/de/pdf-a/fedlex-data-admin-ch-eli-fga-2024-3136-de-pdf-a.pdf
curl -L --fail --retry 3 --retry-delay 2 -o /home/bern/Documents/llm_trustworthiness/data/pdfs/stopfleber_initiative_2024_3077.pdf https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/fga/2024/3077/de/pdf-a/fedlex-data-admin-ch-eli-fga-2024-3077-de-pdf-a-1.pdf
curl -L --fail --retry 3 --retry-delay 2 -o /home/bern/Documents/llm_trustworthiness/data/pdfs/umweltverantwortungsinitiative_2024_109.pdf https://fedlex.data.admin.ch/filestore/fedl

## 7. Save and execute the generated `curl` commands

The exact commands are saved as `scripts/download_selected_botschaften.sh` and then executed from Python. This gives you a standalone shell script as well as a reproducible notebook step.

In [29]:
shell_script_path = SCRIPTS_DIR / "download_selected_botschaften.sh"
script_lines = ["#!/usr/bin/env bash", "set -euo pipefail", ""]
script_lines.extend(record["command"] for record in download_records)
shell_script_path.write_text("\n".join(script_lines) + "\n", encoding="utf-8")
shell_script_path.chmod(0o755)

print(f"Saved curl script: {shell_script_path}")

for record in download_records:
    print(f"\nDownloading: {record['selection_key']}")
    print(record["command"])
    result = subprocess.run(record["cmd"], check=False)
    if result.returncode != 0:
        raise RuntimeError(
            f"curl failed for {record['selection_key']} with exit code {result.returncode}"
        )

print("\nAll six downloads completed successfully.")

Saved curl script: /home/bern/Documents/llm_trustworthiness/scripts/download_selected_botschaften.sh

Downloading: nachhaltigkeitsinitiative
curl -L --fail --retry 3 --retry-delay 2 -o /home/bern/Documents/llm_trustworthiness/data/pdfs/nachhaltigkeitsinitiative_2025_1262.pdf https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/fga/2025/1262/de/pdf-a/fedlex-data-admin-ch-eli-fga-2025-1262-de-pdf-a.pdf

Downloading: neutralitaetsinitiative
curl -L --fail --retry 3 --retry-delay 2 -o /home/bern/Documents/llm_trustworthiness/data/pdfs/neutralitaetsinitiative_2024_3136.pdf https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/fga/2024/3136/de/pdf-a/fedlex-data-admin-ch-eli-fga-2024-3136-de-pdf-a.pdf

Downloading: stopfleber_initiative
curl -L --fail --retry 3 --retry-delay 2 -o /home/bern/Documents/llm_trustworthiness/data/pdfs/stopfleber_initiative_2024_3077.pdf https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/fga/2024/3077/de/pdf-a/fedlex-data-admin-ch-

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100  1.34M 100  1.34M   0      0 19.41M      0                              0
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100 542.4k 100 542.4k   0      0  8.27M      0                              0
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
  0      0   0      0   0      0      0      0                              0


Downloading: umweltverantwortungsinitiative
curl -L --fail --retry 3 --retry-delay 2 -o /home/bern/Documents/llm_trustworthiness/data/pdfs/umweltverantwortungsinitiative_2024_109.pdf https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/fga/2024/109/de/pdf-a/fedlex-data-admin-ch-eli-fga-2024-109-de-pdf-a.pdf

Downloading: steuergerechtigkeits_initiative
curl -L --fail --retry 3 --retry-delay 2 -o /home/bern/Documents/llm_trustworthiness/data/pdfs/steuergerechtigkeits_initiative_2024_589.pdf https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/fga/2024/589/de/pdf-a/fedlex-data-admin-ch-eli-fga-2024-589-de-pdf-a.pdf


100 460.9k 100 460.9k   0      0  6.92M      0                              0
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100 492.8k 100 492.8k   0      0  6.24M      0                              0
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100  1.88M 100  1.88M   0      0 22.54M      0                              0
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
  0      0   0      0   0      0      0      0                              0


Downloading: 13_ahv_rente
curl -L --fail --retry 3 --retry-delay 2 -o /home/bern/Documents/llm_trustworthiness/data/pdfs/13_ahv_rente_2022_1485.pdf https://fedlex.data.admin.ch/filestore/fedlex.data.admin.ch/eli/fga/2022/1485/de/pdf-a/fedlex-data-admin-ch-eli-fga-2022-1485-de-pdf-a.pdf

All six downloads completed successfully.


100  1.38M 100  1.38M   0      0  9.02M      0                              0


## 8. Verify the downloaded PDFs

A successful HTTP request does not by itself guarantee that a usable PDF was saved. The following checks confirm that all six files exist, are non-empty, and begin with the PDF magic bytes `%PDF`.

In [30]:
verification = []

for record in download_records:
    path = Path(record["output_path"])
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    is_pdf_header = False

    if exists and size >= 4:
        with path.open("rb") as f:
            is_pdf_header = f.read(4) == b"%PDF"

    verification.append({
        "selection_key": record["selection_key"],
        "filename": path.name,
        "exists": exists,
        "size_bytes": size,
        "valid_pdf_header": is_pdf_header,
    })

verification_df = pd.DataFrame(verification)
display(verification_df)

if not verification_df["exists"].all():
    raise FileNotFoundError("At least one expected PDF was not downloaded.")
if not verification_df["valid_pdf_header"].all():
    raise ValueError("At least one downloaded file does not have a PDF header.")

,selection_key,filename,exists,size_bytes,valid_pdf_header
0,nachhaltigkeitsinitiative,nachhaltigkeitsinitiative_2025_1262.pdf,True,1410350,True
1,neutralitaetsinitiative,neutralitaetsinitiative_2024_3136.pdf,True,555475,True
2,stopfleber_initiative,stopfleber_initiative_2024_3077.pdf,True,472024,True
3,umweltverantwortungsinitiative,umweltverantwortungsinitiative_2024_109.pdf,True,504628,True
4,steuergerechtigkeits_initiative,steuergerechtigkeits_initiative_2024_589.pdf,True,1975949,True
5,13_ahv_rente,13_ahv_rente_2022_1485.pdf,True,1454249,True


## 9. Install/check the PDF extraction dependency

The extraction stage uses **PyMuPDF** (`fitz`) because it preserves page boundaries. Page-level text will be useful later when source evidence needs to be traced back to the authoritative Botschaft.

If the server environment does not have PyMuPDF installed, uncomment the `%pip` command and run it once.

In [31]:
# Uncomment if necessary:
# %pip install pymupdf

try:
    import fitz  # PyMuPDF
except ImportError as exc:
    raise ImportError(
        "PyMuPDF is not installed. Run `%pip install pymupdf` and rerun this cell."
    ) from exc

print("PyMuPDF imported successfully.")

PyMuPDF imported successfully.


## 10. Extract page-level text from each PDF

Each PDF is extracted page-by-page. For every page we retain the 1-indexed page number, extracted text, and character count.

This preserves a useful evidence unit for later evaluation: a claim can eventually be linked to a specific document and page rather than only to a large undifferentiated document string.

In [32]:
def extract_pdf_pages(pdf_path: Path):
    pages = []
    with fitz.open(pdf_path) as doc:
        metadata = doc.metadata
        page_count = len(doc)
        for page_index, page in enumerate(doc):
            text = page.get_text("text")
            pages.append({
                "page": page_index + 1,
                "text": text,
                "char_count": len(text),
            })
    return metadata, page_count, pages


extraction_records = []
for record in download_records:
    pdf_path = Path(record["output_path"])
    metadata, page_count, pages = extract_pdf_pages(pdf_path)
    extraction_records.append({
        **record,
        "pdf_metadata": metadata,
        "page_count": page_count,
        "pages": pages,
    })

for record in extraction_records:
    total_chars = sum(p["char_count"] for p in record["pages"])
    print(f"{record['selection_key']}: {record['page_count']} pages, {total_chars:,} characters")

nachhaltigkeitsinitiative: 110 pages, 297,961 characters
neutralitaetsinitiative: 30 pages, 80,317 characters
stopfleber_initiative: 16 pages, 43,896 characters
umweltverantwortungsinitiative: 20 pages, 49,954 characters
steuergerechtigkeits_initiative: 108 pages, 273,949 characters
13_ahv_rente: 28 pages, 74,462 characters


## 11. Basic extraction-quality checks

A PDF can download successfully but contain little or no machine-readable text. The checks below flag suspiciously empty documents/pages.

No OCR is silently applied here. If a document is flagged, inspect it first; OCR can be added as a separate, documented preprocessing step if necessary.

In [33]:
quality_rows = []
for record in extraction_records:
    total_chars = sum(p["char_count"] for p in record["pages"])
    empty_pages = sum(p["char_count"] == 0 for p in record["pages"])
    short_pages = sum(p["char_count"] < 100 for p in record["pages"])
    quality_rows.append({
        "selection_key": record["selection_key"],
        "title": record["title"],
        "page_count": record["page_count"],
        "total_characters": total_chars,
        "empty_pages": empty_pages,
        "pages_under_100_chars": short_pages,
        "status": "CHECK" if total_chars < 10000 else "OK",
    })

quality_df = pd.DataFrame(quality_rows)
display(quality_df)

,selection_key,title,page_count,total_characters,empty_pages,pages_under_100_chars,status
0,nachhaltigkeitsinitiative,Botschaft zur Volksinitiative «Keine 10-Millio...,110,297961,0,0,OK
1,neutralitaetsinitiative,Botschaft zur Volksinitiative «Wahrung der sch...,30,80317,0,1,OK
2,stopfleber_initiative,Botschaft zur Volksinitiative «Ja zum Importve...,16,43896,0,0,OK
3,umweltverantwortungsinitiative,Botschaft zur Volksinitiative «Für eine verant...,20,49954,0,1,OK
4,steuergerechtigkeits_initiative,Botschaft zur Volksinitiative «Für eine zivils...,108,273949,0,1,OK
5,13_ahv_rente,Botschaft zur Volksinitiative «Für ein bessere...,28,74462,0,0,OK


## 12. Save the extracted corpus

Three representations are saved:

1. **JSON per document** — structured metadata plus page-level text; this is the master representation.
2. **TXT per document** — convenient human-readable text with explicit page markers.
3. **Corpus manifest CSV** — document-level metadata and extraction statistics.

The original PDFs remain in `data/pdfs/`.

In [35]:
corpus_manifest_rows = []

for record in extraction_records:
    key = record["selection_key"]
    base_name = Path(record["output_path"]).stem
    json_path = TEXT_DIR / f"{base_name}.json"
    txt_path = TEXT_DIR / f"{base_name}.txt"
    selected_row = selected.loc[selected["selection_key"] == key].iloc[0]

    json_payload = {
        "selection_key": key,
        "title": record["title"],
        "date": selected_row["date"],
        "doc": record["doc"],
        "pdfURL": record["pdfURL"],
        "pdf_file": Path(record["output_path"]).name,
        "pdf_metadata": record["pdf_metadata"],
        "page_count": record["page_count"],
        "pages": record["pages"],
        "extracted_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    json_path.write_text(json.dumps(json_payload, ensure_ascii=False, indent=2), encoding="utf-8")

    txt_parts = [
        f"TITLE: {record['title']}",
        f"DOCUMENT URI: {record['doc']}",
        f"PDF URL: {record['pdfURL']}",
        "",
    ]
    for page in record["pages"]:
        txt_parts.append(f"\n===== PAGE {page['page']} =====\n")
        txt_parts.append(page["text"])
    txt_path.write_text("\n".join(txt_parts), encoding="utf-8")

    total_chars = sum(p["char_count"] for p in record["pages"])
    corpus_manifest_rows.append({
        "selection_key": key,
        "title": record["title"],
        "date": selected_row["date"],
        "doc": record["doc"],
        "pdfURL": record["pdfURL"],
        "pdf_file": str(Path(record["output_path"]).resolve().relative_to(Path.cwd().resolve().parent)),
        "json_file": str(json_path.resolve().relative_to(Path.cwd().resolve().parent)),
        "txt_file": str(txt_path.resolve().relative_to(Path.cwd().resolve().parent)),
        "page_count": record["page_count"],
        "total_characters": total_chars,
    })

corpus_manifest = pd.DataFrame(corpus_manifest_rows)
corpus_manifest_path = MANIFEST_DIR / "corpus_manifest.csv"
corpus_manifest.to_csv(corpus_manifest_path, index=False, encoding="utf-8")

print(f"Saved corpus manifest: {corpus_manifest_path}")
print(f"Saved extracted files to: {TEXT_DIR}")

Saved corpus manifest: /home/bern/Documents/llm_trustworthiness/data/manifests/corpus_manifest.csv
Saved extracted files to: /home/bern/Documents/llm_trustworthiness/data/extracted_text


## 13. Final corpus summary

This table is the hand-off from Notebook 01 to the next stage. The next notebook can load `data/manifests/corpus_manifest.csv` and the corresponding JSON files without repeating document discovery, downloading, or extraction.

In [36]:
corpus_manifest[[
    "selection_key", "date", "title", "page_count", "total_characters",
    "pdf_file", "json_file", "txt_file"
]]

,selection_key,date,title,page_count,total_characters,pdf_file,json_file,txt_file
0,nachhaltigkeitsinitiative,2025-04-11,Botschaft zur Volksinitiative «Keine 10-Millio...,110,297961,data/pdfs/nachhaltigkeitsinitiative_2025_1262.pdf,data/extracted_text/nachhaltigkeitsinitiative_...,data/extracted_text/nachhaltigkeitsinitiative_...
1,neutralitaetsinitiative,2024-12-18,Botschaft zur Volksinitiative «Wahrung der sch...,30,80317,data/pdfs/neutralitaetsinitiative_2024_3136.pdf,data/extracted_text/neutralitaetsinitiative_20...,data/extracted_text/neutralitaetsinitiative_20...
2,stopfleber_initiative,2024-12-12,Botschaft zur Volksinitiative «Ja zum Importve...,16,43896,data/pdfs/stopfleber_initiative_2024_3077.pdf,data/extracted_text/stopfleber_initiative_2024...,data/extracted_text/stopfleber_initiative_2024...
3,umweltverantwortungsinitiative,2024-01-19,Botschaft zur Volksinitiative «Für eine verant...,20,49954,data/pdfs/umweltverantwortungsinitiative_2024_...,data/extracted_text/umweltverantwortungsinitia...,data/extracted_text/umweltverantwortungsinitia...
4,steuergerechtigkeits_initiative,2024-03-15,Botschaft zur Volksinitiative «Für eine zivils...,108,273949,data/pdfs/steuergerechtigkeits_initiative_2024...,data/extracted_text/steuergerechtigkeits_initi...,data/extracted_text/steuergerechtigkeits_initi...
5,13_ahv_rente,2022-06-21,Botschaft zur Volksinitiative «Für ein bessere...,28,74462,data/pdfs/13_ahv_rente_2022_1485.pdf,data/extracted_text/13_ahv_rente_2022_1485.json,data/extracted_text/13_ahv_rente_2022_1485.txt


## 14. Reproducibility notes

- The six selected documents are explicitly defined.
- Their PDF URLs are obtained from the existing SPARQL result CSV.
- The original Fedlex document URI is retained.
- The exact generated `curl` commands are saved in `scripts/download_selected_botschaften.sh`.
- The original PDFs are retained.
- Page boundaries are preserved during extraction.
- Structured JSON and human-readable TXT representations are saved.

### Not performed here

This notebook does **not** generate questions, generate answers, score answers, perform LLM-as-a-judge evaluation, or modify source text. Those are later stages.

### Methodological principle

The German Botschaften remain the authoritative source corpus. Any later LLM-generated question must be checked against these source documents before entering the final evaluation dataset.